# Dwarf Example 05: Outer Gap Distribution

**EPS Research — Dwarf/Irregular HI Corpus v1.0**

Outer gap = V_adj(R2) - V_bary(R2). All 24 negative.

**Corpus:** Flynn (2026), Zenodo DOI: 10.5281/zenodo.20320362  
**Sources:** LVHIS (Koribalski 2019), VLA-ANGST (Ott 2012), LITTLE THINGS (Oh 2015), WALLABY DR2  
**Dependencies:** Python 3, numpy, matplotlib

In [ ]:
import json, numpy as np, matplotlib.pyplot as plt
with open('dwarf_irregular_corpus_v1.json') as f:
    corpus = json.load(f)
omega_ready = [g for g in corpus['galaxies']
               if g.get('omega_ready') and g.get('data') and len(g['data'])>=2]
gaps = []
for g in omega_ready:
    d = g['data']
    R=[p['Rad'] for p in d]; V=[p.get('Vrot', 0) for p in d]
    R1,V1=R[0],V[0]; R2,V2=R[-1],V[-1]
    if R1>0 and R2>0 and V1>0 and V2>0:
        omega=(V2/R2-V1/R1)*(R1/R2)**1.5
        gaps.append((V2-R2*omega)-V2)
print(f"All gaps negative: {all(g<0 for g in gaps)}")
print(f"Mean: {np.mean(gaps):.1f} km/s  Std: {np.std(gaps):.1f} km/s")
fig,ax=plt.subplots(figsize=(7,4))
ax.hist(gaps,bins=12,color='#d62728',alpha=0.8,edgecolor='white')
ax.axvline(0,color='black',ls='--',lw=1.5,label='Gap=0')
ax.set_xlabel('Outer gap (km/s)',fontsize=11); ax.set_ylabel('N',fontsize=11)
ax.set_title('Outer Gap Distribution — 24 Omega-Ready Dwarfs',fontsize=10)
ax.legend(); plt.tight_layout()
plt.savefig('dw05_outer_gap.png',dpi=150,bbox_inches='tight'); plt.show()

In [ ]:
import json

with open('/home/david/Documents/RAG Project/Z=2 RAG/Zenodo/intz_corpus_v1b.json') as f:
    data = json.load(f)

galaxies = data.get('galaxies', data)
g522 = next(g for g in galaxies if g['identifiers'].get('kid')==522
            or g['identifiers'].get('id')==522)

print("R_half_kpc:", g522['morphology'].get('R_half_kpc'))
print("stellar_properties:", json.dumps(g522.get('stellar_properties',{}), indent=2))
print("omega:", g522['omega'].get('omega_value_rad_gyr'))
print("RAG:", g522.get('rag_summary','')[:200])

In [ ]:
import json

with open('/home/david/Documents/RAG Project/Z=2 RAG/Zenodo/intz_corpus_v1b.json') as f:
    data = json.load(f)

galaxies = data.get('galaxies', data)

targets = {
    'KID_141': lambda g: g['identifiers'].get('kid')==141 or g['identifiers'].get('id')==141,
    'GS4_33971': lambda g: g['identifiers'].get('kid')=='GS4_33971' or g['identifiers'].get('id')=='GS4_33971' or g['identifiers'].get('name')=='GS4_33971',
    'KID_522': lambda g: g['identifiers'].get('kid')==522 or g['identifiers'].get('id')==522,
}

for label, fn in targets.items():
    g = next((g for g in galaxies if fn(g)), None)
    if not g:
        print(f"\n=== {label} NOT FOUND ===")
        continue
    print(f"\n=== {label} ===")
    print(f"  ID:       {g['identifiers']}")
    print(f"  z_spec:   {g['redshift'].get('z_spec')}")
    print(f"  R_half:   {g['morphology'].get('R_half_kpc')} kpc")
    print(f"  log M*:   {g['stellar_properties'].get('log_mass_msun')}")
    print(f"  SFR:      {g['stellar_properties'].get('sfr_msun_yr')}")
    print(f"  Vc:       {g['kinematics'].get('Vc_kms')} km/s")
    print(f"  sigma0:   {g['kinematics'].get('sigma0_kms')} km/s")
    print(f"  v/sigma:  {g['kinematics'].get('v_over_sigma')}")
    print(f"  omega:    {g['omega'].get('omega_value_rad_gyr')} rad/Gyr")
    print(f"  RAG:      {g.get('rag_summary','')[:150]}")

In [ ]:
import json
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

with open('/home/david/Documents/RAG Project/Z=2 RAG/Zenodo/intz_corpus_v1b.json') as f:
    data = json.load(f)
galaxies = data.get('galaxies', data)

# Extract data
z_all, survey_all, tier_all = [], [], []
omega_vals, omega_z = [], []
vc_vals, sigma_vals, vos_vals = [], [], []
logm_vals, sfr_vals = [], []

for g in galaxies:
    meta = g['metadata']
    red  = g['redshift']
    kin  = g['kinematics']
    om   = g['omega']
    sp   = g.get('stellar_properties', {})
    
    z = red.get('z_spec')
    if z: 
        z_all.append(z)
        survey_all.append(meta['survey'])
        tier_all.append(meta['quality_tier'])
    
    if om.get('omega_available') and om.get('omega_value_rad_gyr') is not None:
        omega_vals.append(om['omega_value_rad_gyr'])
        omega_z.append(z)
    
    vc = kin.get('Vc_kms')
    if vc and vc > 0: vc_vals.append(vc)
    
    s0 = kin.get('sigma0_kms')
    if s0 and s0 > 0 and s0 < 300: sigma_vals.append(s0)
    
    vos = kin.get('v_over_sigma')
    if vos and vos > 0 and vos < 50: vos_vals.append(vos)
    
    lm = sp.get('log_mass_msun')
    if lm: logm_vals.append(lm)
    
    sfr = sp.get('sfr_msun_yr')
    if sfr and sfr > 0: sfr_vals.append(np.log10(sfr))

# Split by survey
z_kross  = [z for z,s in zip(z_all,survey_all) if s=='KROSS']
z_kmos3d = [z for z,s in zip(z_all,survey_all) if s=='KMOS3D']

fig = plt.figure(figsize=(16, 12))
fig.suptitle('EPS Research IntZ Kinematic Corpus v1.0\nKROSS + KMOS³D | 1,292 Galaxies | z~0.4–2.7', 
             fontsize=14, fontweight='bold', y=0.98)

gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# --- Panel 1: Redshift distribution ---
ax1 = fig.add_subplot(gs[0, 0])
ax1.hist(z_kross,  bins=30, alpha=0.7, color='steelblue',  label=f'KROSS (n={len(z_kross)})')
ax1.hist(z_kmos3d, bins=30, alpha=0.7, color='darkorange', label=f'KMOS³D (n={len(z_kmos3d)})')
ax1.set_xlabel('z_spec')
ax1.set_ylabel('N galaxies')
ax1.set_title('Redshift Distribution')
ax1.legend(fontsize=8)

# --- Panel 2: Omega distribution (Tier 1 only) ---
ax2 = fig.add_subplot(gs[0, 1])
ax2.hist(omega_vals, bins=25, color='firebrick', alpha=0.8, edgecolor='darkred')
ax2.axvline(np.median(omega_vals), color='black', linestyle='--', linewidth=2,
            label=f'Median={np.median(omega_vals):.2f}')
ax2.axvline(7.06, color='green', linestyle=':', linewidth=2, label='z=0 SPARC +7.06')
ax2.axvline(0, color='gray', linestyle='-', linewidth=1, alpha=0.5)
ax2.set_xlabel('ω (rad/Gyr)')
ax2.set_ylabel('N galaxies')
ax2.set_title(f'Omega Distribution\n(Tier-1, n={len(omega_vals)}, 100% negative)')
ax2.legend(fontsize=7)

# --- Panel 3: Omega vs redshift ---
ax3 = fig.add_subplot(gs[0, 2])
ax3.scatter(omega_z, omega_vals, c='firebrick', alpha=0.5, s=15)
ax3.axhline(0, color='gray', linestyle='-', linewidth=1, alpha=0.5)
ax3.axhline(7.06, color='green', linestyle=':', linewidth=1.5, label='z=0 SPARC')
ax3.axhline(np.median(omega_vals), color='black', linestyle='--', linewidth=1.5,
            label=f'IntZ median {np.median(omega_vals):.2f}')
# Mark 3 examples
for kid, zv, ov, col, lbl in [
    (141, 0.982, -16.507, 'blue',   'KID-141'),
    (522, 0.824, -33.827, 'purple', 'KID-522'),
]:
    ax3.scatter(zv, ov, c=col, s=80, zorder=5)
    ax3.annotate(lbl, (zv, ov), textcoords='offset points', xytext=(5,5), fontsize=7, color=col)
ax3.set_xlabel('z_spec')
ax3.set_ylabel('ω (rad/Gyr)')
ax3.set_title('Omega vs Redshift (Tier-1)')
ax3.legend(fontsize=7)

# --- Panel 4: Vc distribution ---
ax4 = fig.add_subplot(gs[1, 0])
ax4.hist(vc_vals, bins=30, color='steelblue', alpha=0.8, edgecolor='navy')
ax4.axvline(np.median(vc_vals), color='black', linestyle='--', linewidth=2,
            label=f'Median={np.median(vc_vals):.0f} km/s')
ax4.set_xlabel('Vc (km/s)')
ax4.set_ylabel('N galaxies')
ax4.set_title('Circular Velocity Distribution')
ax4.legend(fontsize=8)

# --- Panel 5: Stellar mass vs SFR (main sequence) ---
ax5 = fig.add_subplot(gs[1, 1])
colors = ['steelblue' if s=='KROSS' else 'darkorange' for s in survey_all 
          if s in ['KROSS','KMOS3D']]
# Just plot all with logm and sfr
logm_kross  = [g['stellar_properties'].get('log_mass_msun') 
               for g in galaxies if g['metadata']['survey']=='KROSS' 
               and g['stellar_properties'].get('log_mass_msun')]
sfr_kross   = [np.log10(g['stellar_properties'].get('sfr_msun_yr',0)+0.001) 
               for g in galaxies if g['metadata']['survey']=='KROSS'
               and g['stellar_properties'].get('log_mass_msun')]
logm_kmos3d = [g.get('photometry',{}).get('log_mstar_msun') or 
               g['stellar_properties'].get('log_mass_msun')
               for g in galaxies if g['metadata']['survey']=='KMOS3D'
               and (g.get('photometry',{}).get('log_mstar_msun') or 
                    g['stellar_properties'].get('log_mass_msun'))]
sfr_kmos3d  = [np.log10(g['stellar_properties'].get('sfr_msun_yr',0)+0.001)
               for g in galaxies if g['metadata']['survey']=='KMOS3D'
               and (g.get('photometry',{}).get('log_mstar_msun') or
                    g['stellar_properties'].get('log_mass_msun'))]

ax5.scatter(logm_kross,  sfr_kross,  c='steelblue',  alpha=0.4, s=8, label='KROSS')
ax5.scatter(logm_kmos3d, sfr_kmos3d, c='darkorange', alpha=0.4, s=8, label='KMOS³D')
ax5.set_xlabel('log M* (Msun)')
ax5.set_ylabel('log SFR (Msun/yr)')
ax5.set_title('Star-Forming Main Sequence')
ax5.legend(fontsize=8)

# --- Panel 6: Three-epoch omega arc ---
ax6 = fig.add_subplot(gs[1, 2])
epochs = [
    (0.05,  +7.06,  3.26,  'SPARC\nz=0\n(84 gal)',    'green'),
    (0.922, -9.087, 8.940, 'IntZ\nz~0.9\n(166 gal)',  'firebrick'),
    (5.0,   -13.05, 3.0,   'ALPINE\nz~5\n(8 gal)',    'purple'),
]
for z, om, sd, lbl, col in epochs:
    ax6.errorbar(z, om, yerr=sd, fmt='o', color=col, markersize=10,
                capsize=5, linewidth=2, label=lbl)
ax6.axhline(0, color='gray', linestyle='-', linewidth=1, alpha=0.5)
ax6.set_xlabel('Redshift z')
ax6.set_ylabel('ω median (rad/Gyr)')
ax6.set_title('Three-Epoch Omega Arc\n(Flynn & Cannaliato 2025)')
ax6.legend(fontsize=7, loc='lower left')
ax6.set_xlim(-0.3, 6)

plt.savefig('/home/david/Documents/RAG Project/Z=2 RAG/Zenodo/intz_corpus_v1b_overview.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: intz_corpus_v1b_overview.png")

In [ ]:
import json

with open('/home/david/Documents/RAG Project/Z=2 RAG/Zenodo/intz_corpus_v1b.json') as f:
    data = json.load(f)

galaxies = data.get('galaxies', data)

# Test all three canonical examples
tests = [
    ('KID-141',    lambda g: g['identifiers'].get('kid')==141),
    ('GS4_33971',  lambda g: g['identifiers'].get('id')=='GS4_33971'),
    ('KID-522',    lambda g: g['identifiers'].get('kid')==522),
]

all_pass = True
for label, fn in tests:
    g = next((g for g in galaxies if fn(g)), None)
    if not g:
        print(f"❌ {label}: NOT FOUND")
        all_pass = False
        continue
    
    # Required fields check
    checks = {
        'survey':       g['metadata'].get('survey'),
        'tier':         g['metadata'].get('quality_tier'),
        'z_spec':       g['redshift'].get('z_spec'),
        'rag_summary':  g.get('rag_summary','')[:50],
        'omega_avail':  g['omega'].get('omega_available'),
        'omega_value':  g['omega'].get('omega_value_rad_gyr'),
        'Vc_kms':       g['kinematics'].get('Vc_kms'),
        'beam_flag':    g['kinematics'].get('beam_smear_corrected'),
        'implausible':  g['kinematics'].get('sigma_implausible_flag','NONE'),
        'outlier_flag': g['omega'].get('omega_outlier_flag','NONE'),
    }
    
    status = "✅ PASS"
    # Fail conditions
    if not checks['z_spec']:          status = "❌ FAIL: missing z_spec"
    if not checks['rag_summary']:     status = "❌ FAIL: missing rag_summary"
    if checks['survey'] not in ['KROSS','KMOS3D']: status = "❌ FAIL: bad survey"
    
    print(f"\n{status} — {label}")
    for k, v in checks.items():
        print(f"  {k:20s}: {v}")

print(f"\n{'='*40}")
print(f"Corpus metadata:")
cm = data.get('corpus_metadata', {})
print(f"  total_galaxies: {cm.get('total_galaxies')}")
print(f"  omega_median:   {cm.get('omega_median_rad_gyr')}")
print(f"  omega_sign:     {cm.get('omega_sign')}")
print(f"  version:        {cm.get('version')}")
print(f"  excluded:       {list(cm.get('excluded_surveys',{}).keys())}")
print(f"\n{'READY FOR ZENODO' if all_pass else 'FIX REQUIRED'}")

In [ ]:
# Cell 1 - Write root README
content = open('/home/david/Downloads/README_github_updated.md').read()